# Resolução Estruturada: Segunda Guerra Mundial (Otimização de Rotas)


Seguindo as orientações dos generais:
- **General Edward:** O risco é diretamente proporcional à distância. Usaremos as distâncias mínimas como pesos do grafo.
- **General Jerzy:** A divisão sobressalente deve permanecer em seu respectivo aquartelamento (custo zero de deslocamento).

Para obter as distâncias mínimas de **todos para todos**, utilizaremos o algoritmo de **Floyd-Warshall** ($O(V^3)$), ideal para grafos pequenos e com múltiplas origens e destinos.

### Mapeamento dos Vértices do Grafo:
* `0`: $\alpha_1$ | `1`: $\alpha_2$ | `2`: $\alpha_3$ (Origens / Aquartelamentos)
* `3`: Nó Intermediário 1 | `4`: Nó Intermediário 2
* `5`: $\beta_1$ | `6`: $\beta_2$ | `7`: $\beta_3$ (Destinos / Frentes de Combate)

In [7]:
import numpy as np

# Número total de vértices no grafo
V = 8
INF = float('inf')

# Inicialização da matriz de adjacência (distâncias diretas extraídas da imagem)
graph = [[INF] * V for _ in range(V)]

# A distância de um nó para ele mesmo é zero
for i in range(V):
    graph[i][i] = 0

# Definição dos arcos baseados no mapa da Figura 1
graph[0][3] = 1  # alpha1 -> Nó 1
graph[1][3] = 2  # alpha2 -> Nó 1
graph[1][4] = 3  # alpha2 -> Nó 2
graph[2][4] = 1  # alpha3 -> Nó 2
graph[2][7] = 4  # alpha3 -> beta3 (Estrada direta)

graph[3][4] = 1  # Nó 1 -> Nó 2
graph[4][3] = 1  # Nó 2 -> Nó 1 (Considerando tráfego bidirecional)

graph[3][5] = 2  # Nó 1 -> beta1
graph[3][6] = 4  # Nó 1 -> beta2
graph[4][5] = 3  # Nó 2 -> beta1
graph[4][6] = 5  # Nó 2 -> beta2
graph[4][7] = 2  # Nó 2 -> beta3

print("Matriz de adjacência inicial configurada com sucesso!")

Matriz de adjacência inicial configurada com sucesso!


### Execução do Algoritmo de Floyd-Warshall

O algoritmo compara todos os caminhos possíveis através de loops aninhados estruturados, atualizando a distância se um nó intermediário `k` oferecer um atalho mais curto entre `i` e `j`:
$$D[i][j] = \min(D[i][j], D[i][k] + D[k][j])$$

In [8]:
# Criar uma cópia da matriz para processamento do algoritmo
dist = [row[:] for row in graph]

# Execução dos 3 loops característicos do Floyd-Warshall
for k in range(V):
    for i in range(V):
        for j in range(V):
            if dist[i][k] + dist[k][j] < dist[i][j]:
                dist[i][j] = dist[i][k] + dist[k][j]

# Exibição elegante da submatriz de interesse (Origens Alpha -> Destinos Beta)
origens = ["alpha_1", "alpha_2", "alpha_3"]
indices_dest = [5, 6, 7]  # Índices correspondentes a beta1, beta2, beta3

print("--- MATRIZ DE DISTÂNCIAS MÍNIMAS EM KM ---")
print(f"{'Origem':<10} | {'beta_1':<8} | {'beta_2':<8} | {'beta_3':<8}")
print("-" * 45)
for i, nome_orig in enumerate(origens):
    line = f"{nome_orig:<10}"
    for j in indices_dest:
        line += f" | {dist[i][j]:<8}"
    print(line)

--- MATRIZ DE DISTÂNCIAS MÍNIMAS EM KM ---
Origem     | beta_1   | beta_2   | beta_3  
---------------------------------------------
alpha_1    | 3        | 5        | 4       
alpha_2    | 4        | 6        | 5       
alpha_3    | 4        | 6        | 3       


### Determinação da Alocação Ótima (Minimização Global)

Analisando os custos mínimos obtidos na matriz anterior:
* **Demandas:** $\beta_1$ precisa de 2 divisões, $\beta_2$ precisa de 1 e $\beta_3$ precisa de 1.
* **Disponibilidade:** $\alpha_1$ possui 2 divisões, $\alpha_2$ possui 2 divisões e $\alpha_3$ possui 1 divisão.

Para obter o menor risco total, a combinação perfeita consiste em utilizar as tropas com menor custo periférico absoluto. Vamos calcular e validar a distribuição ótima.

In [9]:
# Índices mapeados: alpha1=0, alpha2=1, alpha3=2 | beta1=5, beta2=6, beta3=7

# Distribuição Ótima:
# - Enviar as 2 divisões de alpha_1 para beta_1
# - Enviar 1 divisão de alpha_2 para beta_2
# - Enviar 1 divisão de alpha_3 para beta_3
# - Manter 1 divisão restante em alpha_2 (custo zero)

custo_beta1 = 2 * dist[0][5]  # 2 divisões de alpha_1 para beta_1 (2 * 3)
custo_beta2 = 1 * dist[1][6]  # 1 divisão de alpha_2 para beta_2 (1 * 6)
custo_beta3 = 1 * dist[2][7]  # 1 divisão de alpha_3 para beta_3 (1 * 4)

risco_total_minimo = custo_beta1 + custo_beta2 + custo_beta3

print("================ SOLUÇÃO FINAL ================")
print(f"-> Frente beta_1: Recebe 2 divisões vindas de alpha_1. (Distância total: {custo_beta1} km)")
print(f"-> Frente beta_2: Recebe 1 divisão vinda de alpha_2.  (Distância total: {custo_beta2} km)")
print(f"-> Frente beta_3: Recebe 1 divisão vinda de alpha_3.  (Distância total: {custo_beta3} km)")
print("-----------------------------------------------")
print("-> DIVISÃO REMANESCENTE: Permanecerá aquartelada em alpha_2 (Custo: 0 km).")
print("===============================================")
print(f"RISCO DE BOMBARDEAMENTO MÍNIMO TOTAL: {risco_total_minimo} km")

================ SOLUÇÃO FINAL ================
-> Frente beta_1: Recebe 2 divisões vindas de alpha_1. (Distância total: 6 km)
-> Frente beta_2: Recebe 1 divisão vinda de alpha_2.  (Distância total: 6 km)
-> Frente beta_3: Recebe 1 divisão vinda de alpha_3.  (Distância total: 3 km)
-----------------------------------------------
-> DIVISÃO REMANESCENTE: Permanecerá aquartelada em alpha_2 (Custo: 0 km).
RISCO DE BOMBARDEAMENTO MÍNIMO TOTAL: 15 km
